In [11]:
import requests
import pandas as pd
import feedparser
from bs4 import BeautifulSoup
from datetime import datetime
import time
import os

class CyberSecurityArticleFetcher:
    """Fetch cybersecurity articles from multiple sources"""
    
    def __init__(self):
        self.articles = []
        
    def fetch_from_rss_feeds(self):
        """Fetch articles from cybersecurity RSS feeds"""
        feeds = [
            'https://www.bleepingcomputer.com/feed/',
            'https://www.securityweek.com/feed/',
            'https://feeds.feedburner.com/TheHackersNews',
            'https://www.darkreading.com/rss_simple.asp',
            'https://krebsonsecurity.com/feed/',
            'https://www.us-cert.gov/ncas/current-activity.xml',
            'https://www.schneier.com/blog/atom.xml',
            'https://www.troyhunt.com/rss/',
        ]
        
        print("Fetching from RSS feeds...")
        for feed_url in feeds:
            try:
                print(f"  Trying {feed_url}...")
                
                # Set user agent to avoid being blocked
                feedparser.USER_AGENT = "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"
                feed = feedparser.parse(feed_url)
                
                if feed.bozo and not feed.entries:
                    print(f"    ✗ Failed to parse feed or no entries")
                    continue
                
                source = feed.feed.get('title', feed_url.split('/')[2])
                entry_count = min(len(feed.entries), 10)  # Limit to 10 per feed
                
                for entry in feed.entries[:entry_count]:
                    # Get content from different possible fields
                    content = ''
                    if hasattr(entry, 'content') and entry.content:
                        content = self._clean_html(entry.content[0].get('value', ''))
                    elif hasattr(entry, 'description'):
                        content = self._clean_html(entry.description)
                    elif hasattr(entry, 'summary'):
                        content = self._clean_html(entry.summary)
                    
                    # Skip if content is too short
                    if len(content) < 100:
                        continue
                    
                    article = {
                        'title': entry.get('title', 'No Title'),
                        'source': source,
                        'url': entry.get('link', ''),
                        'published_date': entry.get('published', entry.get('updated', datetime.now().isoformat())),
                        'summary': self._clean_html(entry.get('summary', ''))[:200],
                        'content': content
                    }
                    self.articles.append(article)
                    
                print(f"    ✓ Fetched {len([a for a in self.articles if a['source'] == source])} articles from {source}")
                time.sleep(1)  # Be respectful with requests
                
            except Exception as e:
                print(f"    ✗ Error: {str(e)}")
                
        return len(self.articles)
    
    def _clean_html(self, text):
        """Remove HTML tags from text"""
        if not text:
            return ""
        soup = BeautifulSoup(str(text), 'html.parser')
        return soup.get_text(separator=' ', strip=True)
    
    def save_to_csv(self, filename='text_data.csv'):
        """Save articles to CSV file, avoiding duplicates with existing data"""
        if not self.articles:
            print("\n⚠ No articles to save!")
            return None
        
        # Load existing articles if file exists
        existing_articles = []
        existing_titles = set()
        existing_urls = set()
        
        try:
            if os.path.exists(filename) and os.path.getsize(filename) > 0:
                existing_df = pd.read_csv(filename, encoding='utf-8')
                if len(existing_df) > 0:
                    existing_articles = existing_df.to_dict('records')
                    existing_titles = set(existing_df['title'].dropna())
                    existing_urls = set(existing_df['url'].dropna())
                    print(f"📂 Found {len(existing_articles)} existing articles in {filename}")
        except Exception as e:
            print(f"📝 Creating new file {filename} (Error reading existing: {e})")
        
        # Deduplicate new articles and filter against existing ones
        new_articles = []
        duplicates = 0
        
        for article in self.articles:
            title = article.get('title', '')
            url = article.get('url', '')
            
            # Check if article already exists (by title or URL)
            if title in existing_titles or url in existing_urls:
                duplicates += 1
                continue
            
            # Add to tracking sets and new articles list
            if title and url:
                existing_titles.add(title)
                existing_urls.add(url)
                new_articles.append(article)
        
        if duplicates > 0:
            print(f"⚠ Skipped {duplicates} duplicate articles")
        
        if not new_articles and not existing_articles:
            print("\n⚠ No articles to save!")
            return None
        
        if not new_articles and existing_articles:
            print("\n⚠ No new articles to add (all are duplicates)")
            return existing_df
        
        # Create records for new articles
        df_new_records = []
        start_id = len(existing_articles)
        
        for idx, article in enumerate(new_articles):
            # Combine title and content for the text_content field
            full_text = f"{article['title']}. {article['content']}"
            
            df_new_records.append({
                'text_id': start_id + idx,
                'text_content': full_text,
                'title': article['title'],
                'source': article['source'],
                'url': article['url'],
                'published_date': article['published_date'],
                'fetch_date': datetime.now().isoformat()
            })
        
        # Combine existing and new articles
        if existing_articles:
            df_combined = pd.concat([
                pd.DataFrame(existing_articles),
                pd.DataFrame(df_new_records)
            ], ignore_index=True)
        else:
            df_combined = pd.DataFrame(df_new_records)
        
        # Save to CSV
        df_combined.to_csv(filename, index=False, encoding='utf-8')
        print(f"\n💾 Successfully saved to {filename}")
        
        # Verify file was written
        if os.path.exists(filename) and os.path.getsize(filename) > 0:
            print(f"✓ File verified: {os.path.getsize(filename)} bytes")
        else:
            print(f"⚠ Warning: File may not have been saved correctly")
        
        print(f"\n{'='*60}")
        print(f"✓ Added {len(new_articles)} new articles to {filename}")
        print(f"✓ Total articles in file: {len(df_combined)}")
        print(f"{'='*60}")
        print(f"\nDataset Statistics:")
        print(f"  - New articles: {len(new_articles)}")
        print(f"  - Total articles: {len(df_combined)}")
        print(f"  - Total characters: {df_combined['text_content'].str.len().sum():,}")
        print(f"  - Average article length: {df_combined['text_content'].str.len().mean():.0f} characters")
        print(f"  - Sources: {df_combined['source'].nunique()}")
        
        return df_combined

# Initialize fetcher
fetcher = CyberSecurityArticleFetcher()

print("="*60)
print("CYBERSECURITY ARTICLE FETCHER")
print("="*60)

# Fetch from RSS feeds
articles_from_rss = fetcher.fetch_from_rss_feeds()
print(f"\nTotal articles fetched from RSS: {articles_from_rss}")

print(f"\n{'='*60}")
print(f"Total articles collected: {len(fetcher.articles)}")
print(f"{'='*60}")

# Save all articles to CSV
if len(fetcher.articles) > 0:
    df_articles = fetcher.save_to_csv('text_data.csv')
    
    # Display first few articles
    if df_articles is not None and len(df_articles) > 0:
        print("\nSample Articles:")
        display_cols = ['text_id', 'title', 'source']
        print(df_articles[display_cols].head(10).to_string(index=False))
    else:
        print("\n⚠ DataFrame is None or empty")
else:
    print("\n⚠ No articles were fetched. Please check your internet connection or try again later.")

CYBERSECURITY ARTICLE FETCHER
Fetching from RSS feeds...
  Trying https://www.bleepingcomputer.com/feed/...
    ✗ Failed to parse feed or no entries
  Trying https://www.securityweek.com/feed/...
    ✗ Failed to parse feed or no entries
  Trying https://feeds.feedburner.com/TheHackersNews...
    ✗ Failed to parse feed or no entries
  Trying https://www.darkreading.com/rss_simple.asp...
    ✗ Failed to parse feed or no entries
  Trying https://krebsonsecurity.com/feed/...
    ✗ Failed to parse feed or no entries
  Trying https://www.us-cert.gov/ncas/current-activity.xml...
    ✓ Fetched 10 articles from Alerts
  Trying https://www.schneier.com/blog/atom.xml...
    ✗ Failed to parse feed or no entries
  Trying https://www.troyhunt.com/rss/...
    ✗ Failed to parse feed or no entries

Total articles fetched from RSS: 10

Total articles collected: 10
📂 Found 10 existing articles in text_data.csv
⚠ Skipped 10 duplicate articles

⚠ No new articles to add (all are duplicates)

Sample Articles

In [12]:
import re
import pandas as pd
import json
from datetime import datetime

# Load raw text data from CSV
df = pd.read_csv('text_data.csv')

# Text processing function
def process_text(text):
    """
    Process and clean text data.
    
    Args:
        text: Raw text string
    
    Returns:
        Dictionary with processed text and metadata
    """
    # Remove extra whitespace
    cleaned_text = re.sub(r'\s+', ' ', text).strip()
    
    # Extract sentences
    sentences = re.split(r'(?<=[.!?])\s+', cleaned_text)
    sentences = [s.strip() for s in sentences if s.strip()]
    
    # Calculate basic statistics
    word_count = len(cleaned_text.split())
    char_count = len(cleaned_text)
    sentence_count = len(sentences)
    
    # Extract potential keyphrases (simple extraction of common multi-word phrases)
    potential_phrases = re.findall(r'\b[a-z]+(?:\s+[a-z]+){1,3}\b', cleaned_text.lower())
    
    return {
        'original_text': text,
        'cleaned_text': cleaned_text,
        'sentences': sentences,
        'metadata': {
            'word_count': word_count,
            'char_count': char_count,
            'sentence_count': sentence_count,
            'avg_words_per_sentence': round(word_count / sentence_count, 2) if sentence_count > 0 else 0
        }
    }

# Process all texts
processed_data = []
for idx, row in df.iterrows():
    processed_entry = process_text(row['text_content'])
    processed_entry['text_id'] = int(row['text_id'])
    processed_entry['processing_timestamp'] = datetime.now().isoformat()
    processed_data.append(processed_entry)

# Create JSON library structure
text_library = {
    'library_info': {
        'version': '1.0',
        'created_at': datetime.now().isoformat(),
        'total_texts': len(processed_data),
        'source': 'text_data.csv'
    },
    'texts': processed_data
}

# Save to JSON file
with open('text_library.json', 'w', encoding='utf-8') as f:
    json.dump(text_library, f, indent=2, ensure_ascii=False)

print(f"✓ Processed {len(processed_data)} texts")
print(f"✓ Saved to text_library.json")
print(f"\nLibrary structure:")
print(f"  - Total texts: {text_library['library_info']['total_texts']}")
print(f"  - Total words: {sum(t['metadata']['word_count'] for t in processed_data)}")
print(f"  - Total sentences: {sum(t['metadata']['sentence_count'] for t in processed_data)}")

# Display sample entry
print(f"\nSample entry (text_id 0):")
print(json.dumps(processed_data[0], indent=2)[:500] + "...")

✓ Processed 10 texts
✓ Saved to text_library.json

Library structure:
  - Total texts: 10
  - Total words: 6966
  - Total sentences: 318

Sample entry (text_id 0):
{
  "original_text": "MyDoom.B Virus. Systems Affected Any system running Microsoft Windows (Windows 95 and newer) that are used for reading email or accessing peer-to-peer file sharing services. Overview A new variant of the previously discovered MyDoom virus, MyDoom.B, has been identified. In addition to the common traits of email-borne viruses, this virus may prevent your computer from updating anti-virus and other software. Description Quick Links Protect | Identify | Recover Protect Your Sy...


In [17]:
import re
import json
from collections import defaultdict

def batch_search_keyphrases(texts, keyphrases):
    """
    Batch search for key phrases in a list of texts.
    
    Args:
        texts: List of text strings to search
        keyphrases: List of key phrases to search for
    
    Returns:
        Dictionary with keyphrase as key and list of (index, text) tuples as value
    """
    results = defaultdict(list)
    
    # Compile regex patterns for case-insensitive search
    patterns = {phrase: re.compile(re.escape(phrase), re.IGNORECASE) for phrase in keyphrases}
    
    # Batch process all texts
    for idx, text in enumerate(texts):
        for phrase, pattern in patterns.items():
            if pattern.search(text):
                results[phrase].append((idx, text))
    
    return dict(results)

# Load processed texts from JSON library
with open('text_library.json', 'r', encoding='utf-8') as f:
    text_library = json.load(f)

# Extract cleaned texts for searching
texts = [entry['cleaned_text'] for entry in text_library['texts']]

# Split texts into sentences and track which text they came from
text_sentences = []
for idx, entry in enumerate(text_library['texts']):
    for sentence in entry['sentences']:
        text_sentences.append((idx, sentence.strip()))

keyphrases = ["exploit this vulnerability by convincing you","Firewall-1"]

# Search for keyphrases in sentences
search_results = batch_search_keyphrases([s[1] for s in text_sentences], keyphrases)

# Map results back to original text index and sentence
results_with_context = {}
for phrase, matches in search_results.items():
    results_with_context[phrase] = [(text_sentences[idx][0], text_sentences[idx][1]) for idx, _ in matches]

# Also perform full text search
full_text_results = batch_search_keyphrases(texts, keyphrases)

print("=" * 60)
print("SENTENCE-LEVEL SEARCH RESULTS:")
print("=" * 60)
for phrase, matches in results_with_context.items():
    print(f"\n'{phrase}' found in {len(matches)} sentences:")
    for text_id, sentence in matches:
        print(f"  [Text {text_id}] {sentence}")

print("\n" + "=" * 60)
print("FULL TEXT SEARCH RESULTS:")
print("=" * 60)
for phrase, matches in full_text_results.items():
    print(f"\n'{phrase}' found in {len(matches)} texts:")
    for text_id, text in matches:
        print(f"  [Text {text_id}] {text[:100]}...")

SENTENCE-LEVEL SEARCH RESULTS:

'exploit this vulnerability by convincing you' found in 1 sentences:
  [Text 1] The attacker could exploit this vulnerability by convincing you, the victim, to access a specially crafted HTML document such as a web page or HTML email message.

'Firewall-1' found in 6 sentences:
  [Text 2] HTTP Parsing Vulnerabilities in Check Point Firewall-1.
  [Text 2] Systems Affected Check Point Firewall-1 NG FCS Check Point Firewall-1 NG FP1 Check Point Firewall-1 NG FP2 Check Point Firewall-1 NG FP3, HF2 Check Point Firewall-1 NG with Application Intelligence R54 Check Point Firewall-1 NG with Application Intelligence R55 Overview Several versions of Check Point Firewall-1 contain a vulnerability that allows remote attackers to execute arbitrary code with administrative privileges.
  [Text 2] Description The Application Intelligence (AI) component of Check Point Firewall-1 is an application proxy that scans traffic for application layer attacks once it has passed t